In [1]:
import pandas as pd
import requests
import re
from datasets import load_dataset
from datetime import datetime

c:\Users\yagub\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("gsm8k", "main")

test_data = dataset["test"].select(range(5))

In [3]:
def query_ollama(prompt, model="mistral"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0}
        }
    )
    return response.json()["response"]

In [4]:
def extract_correct_answer(answer_text):
    match = re.search(r"####\s*(.*)", answer_text)
    if match:
        return match.group(1).strip()
    return None

In [5]:
def extract_model_answer(text):
    numbers = re.findall(r"-?\d+\.?\d*", text)
    if numbers:
        return numbers[-1]
    return None

In [ ]:
results = []

for i, sample in enumerate(test_data):
    print(f"Processing sample {i+1}")
    
    question = sample["question"]
    correct = extract_correct_answer(sample["answer"])
    
    prompt = f"""
Solve the following math problem.
Give only the final numeric answer.

{question}
"""
    
    raw_output = query_ollama(prompt, model="mistral")
    model_answer = extract_model_answer(raw_output)
    
    is_correct = (model_answer == correct)
    
    results.append({
        "question_id": i,
        "correct_answer": correct,
        "model_answer": model_answer,
        "is_correct": is_correct,
        "timestamp": datetime.now()
    })

Processing sample 1
